# comparison

In [1]:
from pathlib import Path

from combra import angles
from combra.metrics import compare_folders, find_kimg_parquets

# Generated h5s store class_0/1/2; the real ethalon stores class_Ultra_Co*.
# san and diffit indexed their training classes in different orders.
CLASS_MAP = {
    'san':    {'class_0': 'class_Ultra_Co25', 'class_1': 'class_Ultra_Co11', 'class_2': 'class_Ultra_Co6_2'},
    'diffit': {'class_0': 'class_Ultra_Co11', 'class_1': 'class_Ultra_Co25', 'class_2': 'class_Ultra_Co6_2'},
}
GRAIN_LABEL = {'class_Ultra_Co11': 'small grain',
               'class_Ultra_Co25': 'medium grain',
               'class_Ultra_Co6_2': 'large grain'}

# Shared canonical ethalon — same parquet the grid notebook reads (256x256, all steps).
# angles_out_dir applies the same `_msl{int}` folder convention as generation.
GRID = Path('./data/angles')
MIN_SEG_LEN = 5.0
ethalon_path = angles.angles_out_dir(
    GRID, 'o_bc_left_4x_1536_1024x1024_256x256_rgb_N360', MIN_SEG_LEN) / 'angles_n360.parquet'

# Discover this run's per-checkpoint parquets straight from disk: the fully-trained
# N10000 row (reference) followed by every kimg snapshot, chronological — so new
# snapshots are picked up automatically instead of being hand-listed.
RUN = '00017-diffit-256-gpus2-batch192'
folder_paths = find_kimg_parquets(GRID, RUN, n=1000, msl=MIN_SEG_LEN, final_tag='N10000')

# Generated parquet uses class_0/1/2; ethalon uses class_Ultra_Co* (shared map).
class_map = CLASS_MAP['diffit']

# Image-feature metrics (FID / CMMD / FD-DINOv2) are computed here from the h5
# image caches in data/h5 — no longer read from the training stats.jsonl. The
# ethalon's images are the real 256x256 h5; each checkpoint's gen images are the
# kimg/N10000 h5 sharing that folder's stem. compare_folders runs
# compute_all_metrics per (gen->real) class and merges the three image metrics
# into every record (and prints them as columns). CMMD needs the
# `combra[gen-metrics]` extra (open_clip); without it that column nan-fills.
H5 = GRID.parent / 'h5'
real_h5 = str(H5 / 'o_bc_left_4x_1536_1024x1024_256x256_rgb_N360.h5')
gen_h5_map = {str(p): str(H5 / f"{p.parent.name.replace(f'_msl{int(MIN_SEG_LEN)}', '')}.h5")
              for p in folder_paths}

# wdist is angle-space EMD in degrees; coef=1 prints raw degrees. gen_n caps the
# generated images per class to the parquet's N so the N10000 reference row does
# not load all 10k images for the image metrics.
records = compare_folders(folder_paths, str(ethalon_path), class_map=class_map,
                          steps=[2], coef=1, verbose=True,
                          real_h5=real_h5, gen_h5_map=gen_h5_map,
                          image_metrics=True, gen_n=1000)

kimg       class                step  w1   mu1%    mu2%   sigma1% sigma2%   amp1%   amp2%       FID     CMMD  FD-DINO
---------------------------------------------------------------------------------------------------------------------


fid failed: deadlock detected by _ModuleLock('torchvision.ops.misc') at 140038920941840
Using cache found in /home/david/.cache/torch/hub/facebookresearch_dinov2_main
/home/david/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/david/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/david/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/home/david/anaconda3/envs/wc-cv/lib/python3.11/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


00017-diffit-256-gpus2-batch192_N10000_msl5 class_Ultra_Co11      2.0       4.253    -1.45    1.53     6.83   -2.17    -1.13   33.31       nan   546.11  1506.47
00017-diffit-256-gpus2-batch192_N10000_msl5 class_Ultra_Co25      2.0      29.697     4.07  -15.09   -13.97  -46.45    19.15  -83.89    474.53   523.56  1674.10
00017-diffit-256-gpus2-batch192_N10000_msl5 class_Ultra_Co6_2     2.0       2.459     1.39   -3.61     8.54  -17.43     1.17   -1.42    203.95   358.88  1235.60

004435_msl5 class_Ultra_Co11      2.0      12.551     0.94   -1.05    10.20   -9.63    -5.72   97.86    300.96   586.00  1662.40
004435_msl5 class_Ultra_Co25      2.0      23.856     4.31  -13.13    -9.69  -40.20    15.80  -67.55    295.88   608.84  1565.79
004435_msl5 class_Ultra_Co6_2     2.0       4.195     1.04   -2.11     8.62   -2.54    -0.79   39.05    167.80   439.63  1196.84

006451_msl5 class_Ultra_Co11      2.0      11.213     0.59   -0.66     9.97   -7.62    -4.98   87.49    365.63   603.74  1584.02

In [ ]:
from combra.metrics import plot_metrics_overlay

# One overlaid figure per class: the six Gaussian-fit relative errors, W-dist and
# the three image-feature metrics (FID / CMMD / FD-DINOv2), all as |metric| on one
# log-scaled axis vs training kimg. The image metrics are read straight from the
# records compare_folders now produces. The N10000 reference row carries a
# non-kimg tag, so drop it from the curves.
ckpt_records = [r for r in records if r['kimg'].split('_')[0].isdigit()]

for cls, grain in GRAIN_LABEL.items():
    fig = plot_metrics_overlay(
        ckpt_records, cls,
        title=f'{grain} ({cls}) — diffit 256×256',
        save_path=f'metrics_overlay_{cls}_step2.png',
    )
    fig.show()

In [ ]:
import pandas as pd

from combra.metrics import compute_all_metrics_vs_n

# Metric-consistency sweep: compute the FULL metric suite (angle W-dist +
# Gaussian-fit relative errors + image-feature FID / CMMD / FD-DINOv2) straight
# from image batches, for each (resolution, generator) over an N sweep, and save
# one tidy parquet. Image-feature metrics nan-fill when their GPU / the
# combra[gen-metrics] extra is unavailable. Heavy: runs Inception / DINOv2 per N.
SOURCES = {
    256: {'real':   './data/h5/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360.h5',
          'san':    './data/h5/gen_san_256x256_N100_000.h5',
          'diffit': './data/h5/00017-diffit-256-gpus2-batch192_N10000.h5'},
    512: {'real':   './data/h5/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360.h5',
          'san':    './data/h5/gen_san_512x512_N100_000.h5',
          'diffit': './data/h5/00018-diffit-512-gpus4-batch256_N10000.h5'},
}
N_SWEEP = [100, 250, 1000, 10000]

rows = []
for res, group in SOURCES.items():
    for kind in ('san', 'diffit'):
        recs = compute_all_metrics_vs_n(
            group['real'], group[kind], CLASS_MAP[kind],
            ns=N_SWEEP, step=2.0,
        )
        rows.extend({'kind': kind, 'resolution': res, **r} for r in recs)

metrics_all = pd.DataFrame.from_records(rows)
metrics_all.to_parquet('all_metrics_vs_n_step2.parquet', index=False)
print(f'wrote {len(metrics_all)} rows -> all_metrics_vs_n_step2.parquet')
metrics_all.head()

# training-curve grid (from tensorboards)

Per-resolution **3×2 grid** straight from the `tensorboards/` event files (one per
model×resolution). Columns are the two models — **san** (left) and **diffit**
(right); rows are the metric families:

1. **physical metrics** — angle W-dist (`w1/w2`, `circular_w1/w2`) and the
   Gaussian-fit relative errors (`mu`, `sigma`, `amp`);
2. **training metrics** — FID / CMMD / FD-DINOv2 (10k);
3. **losses** — model-specific (`G/D` for san, `train/mse/vb` for diffit).

The two runs live on very different axes (diffit ≈28 000 kimg, san ≈1 600 kimg)
and every metric has its own y-scale, so the x-axis is normalized to training
**progress fraction** (`kimg / max kimg`) and each curve is **min-max
normalized** to [0, 1] — the panels show convergence *shape*, not raw values.

In [ ]:
from collections import defaultdict
from pathlib import Path

import numpy as np
from tensorboard.backend.event_processing import event_file_loader
from tensorboard.util import tensor_util

# One tfevents file per (model, resolution). Both models log the same
# Metrics/combra_* scalars; only the loss tags differ between them.
TB = Path('tensorboards')
RUNS = {
    256:  {'diffit': 'diffit-256x256',   'san': 'san-256x256'},
    512:  {'diffit': 'diffit-512-512',   'san': 'san-512-512'},
    1024: {'diffit': 'diffit-1024x1024', 'san': 'san-1024-1024'},
}
# Tag that maps event-step -> training kimg, per model (used to build a common x).
KIMG_TAG = {'diffit': 'Train/kimg', 'san': 'Progress/kimg'}
LOSS_TAGS = {
    'diffit': ['Train/Loss/train', 'Train/Loss/mse', 'Train/Loss/vb'],
    'san':    ['Loss/G/loss', 'Loss/D/loss'],
}
PHYS_TAGS = ['combra_w2',
             'combra_mu1', 'combra_mu2', 'combra_sigma1', 'combra_sigma2',
             'combra_amp1', 'combra_amp2']
TRAIN_TAGS = ['combra_fid10k', 'combra_cmmd10k', 'combra_fd_dinov2_10k']


def read_scalars(path):
    """tag -> (steps, values) arrays sorted by step, from one tfevents file.

    Handles both simple_value and the tensor-encoded scalars TB now writes;
    non-numeric summaries (e.g. the text log) are skipped.
    """
    d = defaultdict(list)
    for ev in event_file_loader.EventFileLoader(str(path)).Load():
        if not ev.HasField('summary'):
            continue
        for v in ev.summary.value:
            val = None
            if v.HasField('simple_value'):
                val = v.simple_value
            elif v.HasField('tensor'):
                try:
                    a = tensor_util.make_ndarray(v.tensor)
                    if a.dtype.kind in 'fiu':
                        val = float(a.reshape(-1)[0])
                except Exception:
                    val = None
            if val is not None:
                d[v.tag].append((ev.step, val))
    return {t: tuple(np.array(x) for x in zip(*sorted(s))) for t, s in d.items()}


def progress_x(scalars, model, steps):
    """Event steps -> training-progress fraction in [0, 1] via the kimg series."""
    ks, kv = scalars[KIMG_TAG[model]]
    return np.interp(steps, ks, kv) / kv.max()


def norm01(y):
    """Standard min-max: map the full [min, max] range to [0, 1]."""
    lo, hi = np.nanmin(y), np.nanmax(y)
    return (y - lo) / (hi - lo + 1e-12)


# Parse every event file once.
tb_data = {res: {m: read_scalars(TB / fname) for m, fname in models.items()}
           for res, models in RUNS.items()}
print({res: {m: len(sc) for m, sc in d.items()} for res, d in tb_data.items()})

In [ ]:
import matplotlib.pyplot as plt

COLS = ['san', 'diffit']  # left column = san, right column = diffit
YLABEL = 'per-curve min–max'
EMA_ALPHA = 0.5  # weight on the new sample; lower = smoother (little smoothing)

# SAN 512x512 collapsed twice: a transient burst mid-training and a terminal
# divergence that never recovers (both in training-progress fraction). For the
# SAN-512 panels only, the mid-training burst is replaced by a straight dashed
# bridge across the window, and the terminal tail is dropped so the curve stops
# before the collapse.
SAN512_CUT_CENTER = (0.42, 0.50)  # mid-training spike -> dashed bridge
SAN512_CUT_TAIL = 0.89            # terminal collapse -> curve stops here


def _label(tag):
    return tag.replace('combra_', '').split('/', 1)[-1]


def ema(y, alpha=EMA_ALPHA):
    """Light exponential moving average for a bit of smoothing."""
    y = np.asarray(y, float)
    out = np.empty_like(y)
    acc = y[0]
    for i in range(len(y)):
        acc = alpha * y[i] + (1 - alpha) * acc
        out[i] = acc
    return out


def _plot_group(ax, sc, model, tags, res, prefix='', bridge_center=True):
    """One color per metric; each curve lightly EMA-smoothed then min-max
    normalized. Curves are drawn (and the single-column legend reads) in sorted
    label order. For SAN 512x512 the terminal collapse tail is dropped, and (on
    the metric panels) the mid-training burst is replaced by a dashed bridge."""
    present = sorted((t for t in tags if f'{prefix}{t}' in sc), key=_label)
    cmap = plt.get_cmap('tab10' if len(present) <= 10 else 'tab20')
    san512 = model == 'san' and res == 512
    for j, tag in enumerate(present):
        steps, vals = sc[f'{prefix}{tag}']
        x = progress_x(sc, model, steps)
        color = cmap(j % cmap.N)
        if san512:
            keep = x < SAN512_CUT_TAIL  # drop the terminal collapse
            x, vals = x[keep], vals[keep]
        if san512 and bridge_center:
            lo, hi = SAN512_CUT_CENTER
            win = (x >= lo) & (x <= hi)
            left, right = np.where(x < lo)[0], np.where(x > hi)[0]
            if win.any() and len(left) and len(right):
                li, ri = left[-1], right[0]  # real points bracketing the window
                vals = vals.copy()
                vals[win] = np.interp(x[win], [x[li], x[ri]], [vals[li], vals[ri]])
                y = norm01(ema(vals))
                solid = y.copy()
                solid[win] = np.nan  # break the solid line over the window
                ax.plot(x, solid, color=color, lw=1.4, alpha=0.9, label=_label(tag))
                ax.plot(x[li:ri + 1], y[li:ri + 1], color=color, lw=1.4, alpha=0.9, ls='--')
                continue
        ax.plot(x, norm01(ema(vals)),
                color=color, lw=1.4, alpha=0.9, label=_label(tag))
    ax.set_ylim(0, 1)  # min-max maps every curve into [0, 1]
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=12)
    ax.legend(fontsize=11, ncol=1, loc='upper right', framealpha=0.85)


for res, models in tb_data.items():
    fig, axes = plt.subplots(3, 2, figsize=(15, 12), sharex=True, sharey=True)
    fig.suptitle(f'{res}×{res} — per-curve min-max normalized', fontsize=17)

    for c, model in enumerate(COLS):
        sc = models[model]
        axes[0, c].set_title(f'{model} — physical metrics', fontsize=14)
        _plot_group(axes[0, c], sc, model, PHYS_TAGS, res, prefix='Metrics/')

        axes[1, c].set_title(f'{model} — training metrics (FID / CMMD / FD-DINOv2)',
                             fontsize=14)
        _plot_group(axes[1, c], sc, model, TRAIN_TAGS, res, prefix='Metrics/')

        axes[2, c].set_title(f'{model} — losses', fontsize=14)
        _plot_group(axes[2, c], sc, model, LOSS_TAGS[model], res, bridge_center=False)
        axes[2, c].set_xlabel('training progress (kimg / max kimg)', fontsize=13)

    for r in range(3):
        axes[r, 0].set_ylabel(YLABEL, fontsize=12)

    fig.tight_layout(rect=[0, 0, 1, 0.98])
    fig.savefig(f'tb_grid_{res}.png', dpi=110)
    fig.show()